# Plot multi-lead predictions for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [1]:
%matplotlib inline
%load_ext autotime

import sys
import importlib as imp
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy as ct
import plots
import compute_predictions

import experiment_settings
import mahalanobis

time: 1.67 s (started: 2023-11-05 11:52:51 -07:00)


In [2]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

exper_class = experiment_settings.Experiments()
exper_class.get_exp_list_short

array(['testcase', 'production_run_OFDXY', 'production_run_OBDXY',
       'test_run_OBDXY', 'test_run_OFDXY', 'test_2023season_OFDV',
       'test_2023season_OBDV'], dtype='<U20')

time: 2.56 ms (started: 2023-11-05 11:52:53 -07:00)


In [13]:
expname = 'test_2023season_OBDV'

exp_inds = [index for index, exp_string in enumerate(exper_class.get_exp_list) if expname in exp_string]
EXP_NAME_VEC = exper_class.get_exp_list[np.min(exp_inds):np.max(exp_inds)+1]
EXP_NAME_VEC

['test_2023season_OBDV_AL12',
 'test_2023season_OBDV_AL24',
 'test_2023season_OBDV_AL36',
 'test_2023season_OBDV_AL48',
 'test_2023season_OBDV_AL60',
 'test_2023season_OBDV_AL72',
 'test_2023season_OBDV_AL84',
 'test_2023season_OBDV_AL96',
 'test_2023season_OBDV_AL108',
 'test_2023season_OBDV_AL120',
 'test_2023season_OBDV_EP12',
 'test_2023season_OBDV_EP24',
 'test_2023season_OBDV_EP36',
 'test_2023season_OBDV_EP48',
 'test_2023season_OBDV_EP60',
 'test_2023season_OBDV_EP72',
 'test_2023season_OBDV_EP84',
 'test_2023season_OBDV_EP96',
 'test_2023season_OBDV_EP108',
 'test_2023season_OBDV_EP120']

time: 2.5 ms (started: 2023-11-05 11:59:32 -07:00)


In [14]:
DATA_PATH = "data/"
FIGURE_PATH = "figures/analysis/"+expname
PREDICTIONS_PATH = "saved_predictions/"+expname

time: 491 µs (started: 2023-11-05 11:59:33 -07:00)


In [15]:
plt.style.use("seaborn-white")
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
plots.set_plot_rc()
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

time: 848 µs (started: 2023-11-05 11:59:34 -07:00)


# Plot Results

In [6]:
storm_dict = {
    "IAN": {"storm_name": "IAN",
            "year": 2022,
            "extent": [-100,-65,5,35],
            "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
            # https://www.nhc.noaa.gov/aboutcone.shtml
            },
    "FIONA": {"storm_name": "FIONA",
              "year": 2022,
              "extent": [-100,-45,10,55],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "IRMA": {"storm_name": "IRMA",
             "year": 2017,
             "extent": [-100,-20,10,40],
             "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
             # https://www.air-worldwide.com/blog/posts/2017/8/the-ever-shrinking-cone-of-uncertainty/
             },
    "NICOLE": {"storm_name": "NICOLE",
               "year": 2022,
               "extent": [-100,-48,20,45],
               "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
               },
    "JULIA": {"storm_name": "JULIA",
              "year": 2022,
              "extent": [-100,-65,5,20],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "NORMAN": {"storm_name": "NORMAN",
                "year": 2018,
                "pred_time": 90306,
                "extent": [195, 360-135, 5, 35],
                "nhc_cone_radius": {0:8, 12:25, 24:40, 36:51, 48:66, 60:93, 72:93, 96:116, 120:151}
                },
    "HARVEY": {"storm_name": "HARVEY",
                "year": 2017,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
                },
    "DORIAN": {"storm_name": "DORIAN",
                "year": 2019,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:41, 36:54, 48:68, 60:102, 72:102, 96:151, 120:198}
                },
    "OTIS": {"storm_name": "OTIS",
                "year": 2023,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200},
                "basin": "EP"
                },
}

time: 1.15 ms (started: 2023-11-05 11:52:56 -07:00)


In [17]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ("OTIS",):#("FIONA", "IAN", "IRMA", "HARVEY", "DORIAN"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in (123,):

        TESTING_YEAR = storm["year"]
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = expname
            df_pred_test = pd.concat([df_pred_test, df], axis=0)
            
        # get the climatology (average distribution parameters for the relevant basin, year, lead time)
        df_pred_test = df_pred_test.reset_index(drop=True)
        basin = storm['basin']
        df_clima = df_pred_test.copy()
        for leadtime in np.arange(12, 120+12, 12):
            # get the mean distribution parameters for this basin, lead time, and all years
            # df_current = df_pred_test[(df_pred_test['ATCFID'].str.contains(basin)) & (df_pred_test['FHOUR'] == leadtime)]
            # or, get the mean distribution parameters for this basin, lead time, and previous year
            df_current = df_pred_test[(df_pred_test['ATCFID'].str.contains(basin)) & (df_pred_test['FHOUR'] == leadtime) & (df_pred_test['YEAR'] == TESTING_YEAR)]
            mu_u, mu_v = 0, 0
            sigma_u = df_current['sigma_u'].mean()
            sigma_v = df_current['sigma_v'].mean()
            rho = df_current['rho'].mean()
            
            inds = df_pred_test.index[(df_pred_test['ATCFID'].str.contains(basin)) & (df_pred_test['FHOUR'] == leadtime) & (df_pred_test['YEAR'] == TESTING_YEAR)].to_list()
            # inds = df_pred_test.index[(df_pred_test['ATCFID'].str.contains(basin)) & (df_pred_test['FHOUR'] == leadtime)].to_list()
            df_clima.loc[inds, ['sigma_u', 'sigma_v', 'rho']] = [sigma_u, sigma_v, rho]

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["YEAR"] == TESTING_YEAR)].copy()
        df = df.sort_values("MMDDHH").reset_index(drop=True)
        forecast_dates = df["MMDDHH"].unique()

        for i, pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["MMDDHH"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["FHOUR"].unique()]
            
            dfc_storm = df_clima.loc[
                (df_clima["NAME"] == storm["storm_name"]) & (df_clima["MMDDHH"] == storm["pred_time"])].copy()
            dfc_storm = dfc_storm[~np.isnan(dfc_storm['sigma_u'])]
            
            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx)+10), -(360-np.max(extx)-10), np.min(exty)-10, np.max(exty)+10]
            
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            
            # plot probability ellipses for climatology
            details = plots.plot_probability_ellipses(
                dfc_storm,
                ax=ax,
                leadtimes=np.arange(12, 120+12, 12),
                contours=(2/3,),
                # extent = storm["extent"],
                extent = storm_extent,
                alpha=0.3,
                vector=True,
                plot_nhc_cone=False,
                colors='yellow'
            )
            
            # plot probability ellipses
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                # extent = storm["extent"],
                extent = storm_extent,
                alpha=.4,
                vector=True,
                plot_nhc_cone=True,
            )
            plt.gca().get_legend().remove()
            # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/prev_year_climatology_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

            # # plot banana cones
            # try:
            #     fig = plt.figure(dpi=150, )
            #     ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            #     details = plots.plot_banana_of_uncertainty(
            #         df_storm=df_storm,
            #         ax=ax,
            #         # extent=storm["extent"],
            #         extent=storm_extent,
            #         vector=True,
            #         colors=("steelblue","khaki"),
            #         alpha=.75,
            #         plot_nhc_cone=True,
            #     )
            #     # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            #     ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            #     plt.savefig(
            #         FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
            #         dpi=dpiFig,
            #         bbox_inches='tight',
            #     )
            #     plt.close()
            # except:
            #     print('not enough data for spline computation. not making the figure.')
            #     plt.close()

OTIS
1 of 10: 102218
2 of 10: 102300
3 of 10: 102306
4 of 10: 102312
5 of 10: 102400
6 of 10: 102406
7 of 10: 102412
8 of 10: 102418
9 of 10: 102500
10 of 10: 102506
time: 4.92 s (started: 2023-11-05 11:59:56 -07:00)


In [18]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ("OTIS",):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in (123,):

        TESTING_YEAR = storm["year"]
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = expname
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["YEAR"] == TESTING_YEAR)].copy()
        df = df.sort_values("MMDDHH").reset_index(drop=True)
        forecast_dates = df["MMDDHH"].unique()

        for i,pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["MMDDHH"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["FHOUR"].unique()]
            
            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx)+5), -(360-np.max(extx)-5), np.min(exty)-5, np.max(exty)+5]
            
            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                # extent = storm["extent"],
                extent = storm_extent,
                alpha=.4,
                vector=True,
                plot_nhc_cone=False,
            )
            plt.gca().get_legend().remove()
            # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

            # plot banana cones
            try:
                fig = plt.figure(dpi=150, )
                ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
                details = plots.plot_banana_of_uncertainty(
                    df_storm=df_storm,
                    ax=ax,
                    # extent=storm["extent"],
                    extent=storm_extent,
                    vector=True,
                    colors=("steelblue","khaki"),
                    alpha=.75,
                    plot_nhc_cone=True,
                )
                # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
                ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
                plt.savefig(
                    FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                    dpi=dpiFig,
                    bbox_inches='tight',
                )
                plt.close()
            except:
                print('not enough data for spline computation. not making the figure.')
                plt.close()

OTIS
1 of 10: 102218
2 of 10: 102300
3 of 10: 102306
4 of 10: 102312
5 of 10: 102400
6 of 10: 102406
7 of 10: 102412
not enough data for spline computation. not making the figure.
8 of 10: 102418
not enough data for spline computation. not making the figure.
9 of 10: 102500
not enough data for spline computation. not making the figure.
10 of 10: 102506
not enough data for spline computation. not making the figure.
time: 8.25 s (started: 2023-11-05 12:00:01 -07:00)
